# BrainX 实战：Mante et al. (2013) PFC 群体动力学分析与 Rate-Model 扩展

## 原始工作背景

Mante et al. (2013) 研究的问题是：前额叶皮层如何在相同感觉输入下，根据任务上下文选择相关信息、抑制无关信息，并完成灵活决策。实验中，猕猴执行 context-dependent decision task；每个 trial 的随机点刺激同时包含 motion 和 color 两个证据维度，context cue 指示当前应依据 motion 还是 color 作出选择。

**数据。** 原始数据来自猕猴 prefrontal cortex / frontal eye field 区域的单元记录。每个记录单元包含 trial-by-time 的神经响应，以及每个 trial 对应的 motion coherence、color coherence、context、choice、correctness 等任务变量。由于这些单元多为 sequential recordings，分析中通常构建 pseudo-population，并按条件平均得到 population response。

**方法。** 原文首先对 PFC 群体活动进行 regression、PCA 去噪和 targeted dimensionality reduction (TDR)，提取 choice、motion、color 和 context 相关的低维任务轴。随后训练一个 nonlinear firing-rate RNN 执行同一 context-dependent integration task；该 RNN 的训练目标是完成任务选择，而不是直接拟合 PFC 单神经元活动。训练后，作者通过 population trajectory、fixed-point analysis 和局部线性化分析解释网络如何实现选择性积分。

**主要结果。** PFC 单神经元表现出复杂的 mixed selectivity，但群体层面可以分离出与任务变量相关的低维轨迹。训练后的 rate RNN 产生了与任务计算相匹配的动力学结构：在不同 context 下形成近似 line attractor，使相关证据沿稳定方向积分，而无关证据主要投影到快速衰减方向。该结果支持一种动力学解释：context 并非简单切换输入通道，而是调制整个 recurrent system 的流场。

## 本 notebook 的目标

这个 notebook 使用 Mante et al. (2013) PFC 数据，先重建原文中 population-dynamics 分析的主要思路，再在 BrainX 平台上扩展一个 rate-based trace-fitting 示例。

1. 读取官方 `.mat` 单元记录并按 20 Hz 重采样。
2. 复现行为 psychometric curve 的基本结构。
3. 构建 condition-averaged population response。
4. 用 PCA 去噪，并用 targeted dimensionality reduction (TDR) 得到 choice / motion / color / context 任务轴。
5. 在 BrainX 中训练任务型 rate RNN，复现 context-dependent integration 的计算目标。
6. 在低维任务空间拟合一个 linear dynamical system (LDS)。
7. 在 BrainX 生态中进一步训练 rate-based RNN，演示神经群体 trace fitting。

默认使用完整 monkey A 数据中的前 200 个 units，以兼顾轨迹质量和运行速度；如需快速演示，可在配置单元切回 `PFC_FAST`。


## 数据链接

https://www.ini.uzh.ch/en/research/groups/mante/data.html

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter1d
from sklearn.decomposition import PCA

import jax
import jax.numpy as jnp

try:
    import brainstate as bst
    import brainunit as u
    BRAINX_AVAILABLE = True
    brainx_jit = bst.transform.jit
    print(f"BrainX/brainstate is available: brainstate {getattr(bst, '__version__', 'unknown')}")
except Exception as exc:
    BRAINX_AVAILABLE = False
    brainx_jit = jax.jit
    warnings.warn(f"BrainX/brainstate import failed, falling back to jax.jit: {exc}")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

np.random.seed(7)

## 1. 路径与数据集选择

目录结构来自官方数据包：

- `data/tdr`: 官方 Matlab TDR 代码和一个小型示例数据子集。
- `data/PFC data/PFC data 1`: monkey A 的较完整数据。
- `data/PFC data/PFC data 2`: monkey F 的较完整数据。

为了保证 notebook 能快速跑通，默认使用 `PFC data 1a`。

In [ ]:
WORK_DIR = Path.cwd()
if not (WORK_DIR / "data").exists() and (WORK_DIR / "2026年4月28日" / "data").exists():
    WORK_DIR = WORK_DIR / "2026年4月28日"

DATA_DIR = WORK_DIR / "data"
TDR_DIR = DATA_DIR / "tdr"
PFC_FAST = TDR_DIR / "PFC data 1a"
PFC_FULL_A = DATA_DIR / "PFC data" / "PFC data 1"
PFC_FULL_F = DATA_DIR / "PFC data" / "PFC data 2"

# 默认使用完整 monkey A 的前 200 个 units，兼顾展示质量和运行速度。
# 快速演示可改为：DATASET_DIR = PFC_FAST; MAX_UNITS = None
# 论文级群体轨迹可改为：DATASET_DIR = PFC_FULL_A; MAX_UNITS = None
DATASET_DIR = PFC_FULL_A
ANIMAL = "ar"
EVENT = "Vstim_100_850_ms"
FD = 20
MAX_UNITS = 200

print("work dir:", WORK_DIR)
print("dataset:", DATASET_DIR)
print("n .mat files:", len(list(DATASET_DIR.glob("*.mat"))))

## 2. 读取官方 `.mat` 单元文件

官方 `tdrLoadPfcData.m` 的关键处理是：

- 每个 `.mat` 文件对应一个 recorded unit。
- `unit.response` 是 trial × time 的 spike 序列。
- 先乘以原始采样率，把 spike/bin 转成 firing rate，再用 50 ms boxcar 窗口下采样到 20 Hz。
- `unit.task_variable` 保存每个 trial 的 motion、color、context、choice、correct 等变量。

下面用 Python 复现这个读取逻辑。

In [ ]:
def matstruct_to_dict(obj):
    """Convert a scipy.io mat_struct into a plain nested dict."""
    if hasattr(obj, "_fieldnames"):
        return {name: matstruct_to_dict(getattr(obj, name)) for name in obj._fieldnames}
    if isinstance(obj, np.ndarray) and obj.dtype == object:
        return np.array([matstruct_to_dict(x) for x in obj.ravel()], dtype=object).reshape(obj.shape)
    return obj


def box_resample_to_hz(response, time, fd=20):
    """Replicate official boxresample(Fs * response, R, R, Fs, t0)."""
    response = np.asarray(response, dtype=float)
    time = np.asarray(time, dtype=float)
    fs = int(round(1.0 / float(time[1] - time[0])))
    box = int(round(fs / fd))
    starts = np.arange(0, response.shape[1] - box + 1, box)
    scaled = fs * response
    out = np.stack([np.nanmean(scaled[:, s:s + box], axis=1) for s in starts], axis=1)
    out_time = starts / fs + box / fs / 2.0 + time[0]
    return out, out_time


def load_unit_file(path, fd=20):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    unit = mat["unit"]
    response, time = box_resample_to_hz(unit.response, unit.time, fd=fd)
    task_variable = matstruct_to_dict(unit.task_variable)
    return {
        "name": str(unit.name),
        "response": response,
        "time": time,
        "task_variable": task_variable,
        "unit_info": matstruct_to_dict(unit.unitInfo),
        "stim_info": matstruct_to_dict(unit.stimInfo),
        "path": Path(path),
    }


def load_population(datadir, animal="ar", event="Vstim_100_850_ms", fd=20, max_units=None):
    paths = sorted(Path(datadir).glob(f"{animal}*{event}.mat"))
    if max_units is not None:
        paths = paths[:max_units]
    units = [load_unit_file(p, fd=fd) for p in paths]
    if len(units) == 0:
        raise FileNotFoundError(f"No unit files found in {datadir}")
    time = units[0]["time"]
    return units, time

units, tt = load_population(DATASET_DIR, animal=ANIMAL, event=EVENT, fd=FD, max_units=MAX_UNITS)
print(f"Loaded {len(units)} units")
print("time bins:", len(tt), "from", round(float(tt[0]), 3), "to", round(float(tt[-1]), 3), "s")
print("example response shape:", units[0]["response"].shape, "trial x time")
print("task variables:", list(units[0]["task_variable"].keys()))

## 3. 行为曲线：context 是否选择相关证据？

Mante 任务的核心是：同一个随机点刺激同时包含 motion 和 color 信息，context cue 决定当前 trial 应依据哪一个维度做选择。

这里先检查行为层面是否能看到这个结构：

- motion context 中，choice 应主要随 motion coherence 改变。
- color context 中，choice 应主要随 color coherence 改变。

In [ ]:
def mean_choice_curve(units, xvar, yvar, context_value):
    levels = np.unique(np.concatenate([np.asarray(unit["task_variable"][xvar]).ravel() for unit in units]))
    curves = []
    for unit in units:
        tv = unit["task_variable"]
        x = np.asarray(tv[xvar])
        y = np.asarray(tv[yvar]) == 1
        ctx = np.asarray(tv["context"]) == context_value
        vals = []
        for lev in levels:
            m = ctx & np.isclose(x, lev)
            vals.append(np.nan if not np.any(m) else np.mean(y[m]))
        curves.append(vals)
    curves = np.asarray(curves, dtype=float)
    return levels, np.nanmean(curves, axis=0), np.nanstd(curves, axis=0) / np.sqrt(np.sum(np.isfinite(curves), axis=0))

fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharey=True)
plots = [
    (axes[0, 0], "stim_dir", "targ_dir", +1, "motion context: choice vs motion"),
    (axes[0, 1], "stim_col", "targ_col", +1, "motion context: choice vs color"),
    (axes[1, 0], "stim_dir", "targ_dir", -1, "color context: choice vs motion"),
    (axes[1, 1], "stim_col", "targ_col", -1, "color context: choice vs color"),
]
for ax, xvar, yvar, ctx, title in plots:
    x, y, sem = mean_choice_curve(units, xvar, yvar, ctx)
    ax.errorbar(x, y, yerr=sem, marker="o", lw=1.8, capsize=2)
    ax.axhline(0.5, color="0.5", lw=1, ls="--")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xvar)
    ax.set_ylabel(f"P({yvar}=+1)")
    ax.set_ylim(-0.05, 1.05)
fig.tight_layout()

## 4. 构建 condition-averaged population response

为了得到群体轨迹，需要把 sequentially recorded units 拼成一个 pseudo-population。这里使用正确 trial，并按：

- context：motion / color
- motion coherence：`stim_dir`
- color evidence toward preferred direction：`stim_col2dir`

分组平均。这样可以得到约 `2 × 6 × 6 = 72` 个条件，与论文分析的条件规模一致。

In [ ]:
def condition_average(units, correct_only=True, min_unit_fraction=0.5):
    motion_levels = np.unique(np.concatenate([np.asarray(u["task_variable"]["stim_dir"]).ravel() for u in units]))
    color_levels = np.unique(np.concatenate([np.asarray(u["task_variable"]["stim_col2dir"]).ravel() for u in units]))
    contexts = np.array([-1, 1], dtype=int)

    conditions = []
    for ctx in contexts:
        for m in motion_levels:
            for c in color_levels:
                choice = np.sign(m) if ctx == 1 else np.sign(c)
                conditions.append({
                    "context": ctx,
                    "stim_dir": float(m),
                    "stim_col2dir": float(c),
                    "targ_dir": int(choice),
                })

    n_units = len(units)
    n_time = len(units[0]["time"])
    n_cond = len(conditions)
    X = np.full((n_units, n_time, n_cond), np.nan, dtype=float)
    n_trials = np.zeros((n_units, n_cond), dtype=int)

    for i, unit in enumerate(units):
        tv = unit["task_variable"]
        response = unit["response"]
        for j, cond in enumerate(conditions):
            mask = (
                (np.asarray(tv["context"]) == cond["context"]) &
                np.isclose(np.asarray(tv["stim_dir"]), cond["stim_dir"]) &
                np.isclose(np.asarray(tv["stim_col2dir"]), cond["stim_col2dir"])
            )
            if correct_only:
                mask &= (np.asarray(tv["correct"]) == 1)
            n_trials[i, j] = int(np.sum(mask))
            if n_trials[i, j] > 0:
                X[i, :, j] = np.nanmean(response[mask], axis=0)

    enough_units = np.sum(np.isfinite(X[:, 0, :]), axis=0) >= max(3, int(np.ceil(min_unit_fraction * n_units)))
    X = X[:, :, enough_units]
    n_trials = n_trials[:, enough_units]
    conditions = [c for c, keep in zip(conditions, enough_units) if keep]

    # Fill missing unit-condition averages with that unit's time-specific mean across available conditions.
    filler = np.nanmean(X, axis=2, keepdims=True)
    X = np.where(np.isfinite(X), X, filler)
    return X, conditions, n_trials

X_hz, conditions, n_trials = condition_average(units, correct_only=True)
print("condition-averaged response shape: units x time x conditions =", X_hz.shape)
print("mean trials per unit-condition:", round(float(np.mean(n_trials[n_trials > 0])), 2))
print("conditions kept:", len(conditions))

In [ ]:
def smooth_and_zscore(X, time, width_s=0.04):
    dt = float(np.median(np.diff(time)))
    sigma_bins = width_s / dt
    Xs = gaussian_filter1d(X, sigma=sigma_bins, axis=1, mode="nearest")
    mu = np.nanmean(Xs, axis=(1, 2), keepdims=True)
    sd = np.nanstd(Xs, axis=(1, 2), keepdims=True)
    sd = np.where(sd < 1e-8, 1.0, sd)
    return (Xs - mu) / sd, mu, sd

X, unit_mean, unit_std = smooth_and_zscore(X_hz, tt, width_s=0.04)

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(np.nanmean(X, axis=2), aspect="auto", cmap="coolwarm", vmin=-1.5, vmax=1.5,
               extent=[tt[0], tt[-1], X.shape[0], 0])
ax.set_title("Mean normalized firing rate across conditions")
ax.set_xlabel("time from dots onset (s)")
ax.set_ylabel("unit")
fig.colorbar(im, ax=ax, label="z-scored rate")
fig.tight_layout()

## 5. PCA 去噪：先确定群体活动的主子空间

官方 TDR 流程先对 condition-averaged population response 做 PCA，并保留前 12 个 PCs 作为去噪子空间。这里在小数据集上保留 `min(12, n_units)` 个 PCs。

In [ ]:
U, T, C = X.shape
X_mat = X.reshape(U, T * C).T  # samples x units
pca = PCA(n_components=min(12, U), random_state=0)
score = pca.fit_transform(X_mat)
pcs = pca.components_.T  # units x PCs

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker="o")
ax.set_xlabel("number of PCs")
ax.set_ylabel("cumulative explained variance")
ax.set_title("PCA denoising subspace")
ax.set_ylim(0, 1.02)
fig.tight_layout()

print("PC subspace shape:", pcs.shape)
print("variance explained by kept PCs:", round(float(np.sum(pca.explained_variance_ratio_)), 3))

## 6. Targeted Dimensionality Reduction (TDR)

TDR 的思想是：不要只找最大方差方向，而是找与任务变量最相关的方向。这里对每个 unit、每个时间点进行线性回归：

\[
r_i(t) = \beta_0(t) + \beta_{choice}(t) d + \beta_{motion}(t)m + \beta_{color}(t)c + \beta_{context}(t)s + \epsilon.
\]

随后把每个变量对应的回归系数向量视为 population state space 中的方向，并投影到 PCA 去噪子空间。

In [ ]:
REGRESSORS = ["targ_dir", "stim_dir", "stim_col2dir", "context"]
REGRESSOR_LABELS = {
    "targ_dir": "choice",
    "stim_dir": "motion",
    "stim_col2dir": "color",
    "context": "context",
}


def design_matrix(task_variable, regressors=REGRESSORS):
    cols = [np.ones_like(np.asarray(task_variable[regressors[0]], dtype=float))]
    for name in regressors:
        v = np.asarray(task_variable[name], dtype=float)
        scale = np.nanmax(np.abs(v))
        cols.append(v / scale if scale > 0 else v)
    return np.column_stack(cols)


def trial_regression(units, regressors=REGRESSORS, ridge=1e-6):
    n_units = len(units)
    n_reg = len(regressors)
    n_time = len(units[0]["time"])
    beta = np.zeros((n_units, n_reg, n_time), dtype=float)
    for i, unit in enumerate(units):
        Y = unit["response"].astype(float)
        Y = (Y - np.nanmean(Y)) / (np.nanstd(Y) + 1e-8)
        D = design_matrix(unit["task_variable"], regressors=regressors)
        lhs = D.T @ D + ridge * np.eye(D.shape[1])
        lhs[0, 0] -= ridge  # do not regularize intercept
        coef = np.linalg.solve(lhs, D.T @ Y)
        beta[i] = coef[1:, :]
    return beta

beta = trial_regression(units)

# Denoise regression vectors by projecting them into the PCA subspace.
beta_denoised = np.zeros_like(beta)
P = pcs
for r in range(beta.shape[1]):
    for t in range(beta.shape[2]):
        beta_denoised[:, r, t] = P @ (P.T @ beta[:, r, t])

fig, axes = plt.subplots(1, 4, figsize=(11, 2.8), sharey=True)
for i, name in enumerate(REGRESSORS):
    norm_t = np.linalg.norm(beta_denoised[:, i, :], axis=0)
    axes[i].plot(tt, norm_t, lw=2)
    axes[i].set_title(REGRESSOR_LABELS[name])
    axes[i].set_xlabel("time (s)")
axes[0].set_ylabel("||denoised beta(t)||")
fig.suptitle("Temporal strength of task-related regression vectors", y=1.05)
fig.tight_layout()

## 7. 定义 choice / motion / color / context 任务轴

与官方脚本一致，取不同时间窗内的回归向量平均，得到四个 task-related vectors。随后对它们做 QR orthogonalization，形成四维任务子空间。

In [ ]:
WINDOWS = {
    "targ_dir": (0.60, 0.90),
    "stim_dir": (0.25, 0.45),
    "stim_col2dir": (0.35, 0.55),
    "context": (0.10, 0.90),
}

vectors = []
for i, name in enumerate(REGRESSORS):
    lo, hi = WINDOWS[name]
    mask = (tt >= lo) & (tt <= hi)
    vectors.append(np.nanmean(beta_denoised[:, i, :][:, mask], axis=1))
V = np.column_stack(vectors)  # units x task variables
Q, R = np.linalg.qr(V)
axes_task = Q[:, :len(REGRESSORS)]

# Align signs so each orthogonalized axis points roughly in the original regression-vector direction.
for i in range(axes_task.shape[1]):
    if np.dot(axes_task[:, i], V[:, i]) < 0:
        axes_task[:, i] *= -1

# Project normalized condition-average responses into the task subspace.
Z = np.einsum("ua,utc->atc", axes_task, X)  # task_dim x time x condition
axis_names = [REGRESSOR_LABELS[n] for n in REGRESSORS]
print("TDR projection shape: task_dim x time x condition =", Z.shape)
print("axes:", axis_names)

In [ ]:
def condition_array(conditions, key):
    return np.array([c[key] for c in conditions])

cond_context = condition_array(conditions, "context")
cond_motion = condition_array(conditions, "stim_dir")
cond_color = condition_array(conditions, "stim_col2dir")


def aggregate_by(ctx_value, group_values, group_key):
    trajs = []
    for val in group_values:
        if group_key == "stim_dir":
            mask = (cond_context == ctx_value) & np.isclose(cond_motion, val)
        elif group_key == "stim_col2dir":
            mask = (cond_context == ctx_value) & np.isclose(cond_color, val)
        else:
            raise ValueError(group_key)
        trajs.append(np.nanmean(Z[:, :, mask], axis=2))
    return np.stack(trajs, axis=2)  # dim x time x level

motion_levels = np.unique(cond_motion)
color_levels = np.unique(cond_color)

Zm_motion_ctx = aggregate_by(+1, motion_levels, "stim_dir")
Zm_color_ctx = aggregate_by(-1, motion_levels, "stim_dir")
Zc_motion_ctx = aggregate_by(+1, color_levels, "stim_col2dir")
Zc_color_ctx = aggregate_by(-1, color_levels, "stim_col2dir")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for k, lev in enumerate(motion_levels):
    color = plt.cm.Greys(0.25 + 0.65 * k / max(1, len(motion_levels)-1))
    axes[0].plot(Zm_motion_ctx[1, :, k], Zm_motion_ctx[0, :, k], "-o", color=color, ms=3, label=f"m={lev:g}")
axes[0].set_title("motion context: choice vs motion axis")
axes[0].set_xlabel("motion axis")
axes[0].set_ylabel("choice axis")
axes[0].legend(fontsize=7, frameon=False)

for k, lev in enumerate(color_levels):
    color = plt.cm.Blues(0.25 + 0.65 * k / max(1, len(color_levels)-1))
    axes[1].plot(Zc_color_ctx[2, :, k], Zc_color_ctx[0, :, k], "-o", color=color, ms=3, label=f"c={lev:g}")
axes[1].set_title("color context: choice vs color axis")
axes[1].set_xlabel("color axis")
axes[1].set_ylabel("choice axis")
axes[1].legend(fontsize=7, frameon=False)

fig.suptitle("TDR population trajectories")
fig.tight_layout()

### 7.1 原论文中的 rate RNN 建模对象

Mante et al. 原文确实使用了 firing-rate RNN，但其目的不是直接拟合 PFC 单神经元 trace，也不是拟合 PCA/TDR 后的神经轨迹。

原文中的 RNN 是一个任务模型：网络接收 color evidence、motion evidence 和 context cue，训练目标是完成 context-dependent integration task，并在 trial 末端输出正确选择。训练完成后，作者分析 RNN 内部动力学，寻找 fixed points、line attractors 和 context-dependent selection vectors，用来解释 PFC 群体活动中观察到的选择性积分机制。

因此，原文的逻辑是：

1. 用真实 PFC 数据做 regression / PCA / TDR，得到任务相关 population trajectories。
2. 训练一个 nonlinear firing-rate RNN 执行相同任务。
3. 用相似的 population-dynamics 分析比较 RNN 与 PFC 数据，并解释网络如何实现“相关证据积分、无关证据抑制”。

本 notebook 前面的 PCA/TDR 部分对应原文的数据分析思路；后面的 rate network 拟合 neural trace 是一个 BrainX 实战扩展，用于演示如何进一步把 rate model 直接用于神经活动 trace fitting。二者应当区分开。

## 8. BrainX 任务型 rate RNN：context-dependent integration

这里构建一个任务型 firing-rate RNN。它的监督目标不是 PFC 单元活动，而是完成 context-dependent integration task。

任务结构：

- 输入包含 color evidence、motion evidence、color-context cue、motion-context cue。
- 在 motion context 中，网络应根据 motion evidence 输出选择。
- 在 color context 中，网络应根据 color evidence 输出选择。
- irrelevant evidence 同时存在，但不应决定输出。

训练完成后，我们可检查两个层面：一是输出是否呈现 context-dependent psychometric behavior；二是内部 rate activity 是否形成可视化的低维轨迹。

In [ ]:
from dataclasses import dataclass
import optax


@dataclass(frozen=True)
class TaskRateRNNConfig:
    n_input: int = 4
    n_hidden: int = 100
    n_output: int = 1
    dt: float = 0.01
    tau: float = 0.05
    gain: float = 1.1


class TaskRateNeuron(bst.nn.Dynamics):
    """Continuous-time rate neuron population for task RNN."""

    def __init__(self, n_hidden, dt, tau):
        super().__init__(n_hidden)
        self.n_hidden = n_hidden
        self.dt = dt
        self.tau = tau

    def init_state(self, batch_size=None, **kwargs):
        shape = (self.n_hidden,) if batch_size is None else (batch_size, self.n_hidden)
        self.x = bst.HiddenState(jnp.zeros(shape, dtype=jnp.float32))

    def step_value(self, x, current):
        x_next = x + self.dt * (-x + current) / self.tau
        r_next = jnp.tanh(x_next)
        return x_next, r_next

    def update(self, current):
        x_next, r_next = self.step_value(self.x.value, current)
        self.x.value = x_next
        return r_next


class ContextIntegrationRNN(bst.nn.Module):
    """Firing-rate RNN trained to solve the context-dependent integration task."""

    def __init__(self, config: TaskRateRNNConfig):
        super().__init__()
        self.config = config
        self.pop = TaskRateNeuron(config.n_hidden, config.dt, config.tau)

    def init_params(self, key):
        cfg = self.config
        k1, k2, k3 = jax.random.split(key, 3)
        return {
            "J": cfg.gain * jax.random.normal(k1, (cfg.n_hidden, cfg.n_hidden)) / jnp.sqrt(float(cfg.n_hidden)),
            "Win": 0.5 * jax.random.normal(k2, (cfg.n_hidden, cfg.n_input)),
            "bias": jnp.zeros((cfg.n_hidden,), dtype=jnp.float32),
            "Wout": 0.02 * jax.random.normal(k3, (cfg.n_output, cfg.n_hidden)),
            "bout": jnp.zeros((cfg.n_output,), dtype=jnp.float32),
        }

    def one_step(self, params, x, u_t):
        r = jnp.tanh(x)
        current = params["J"] @ r + params["Win"] @ u_t + params["bias"]
        x_next, r_next = self.pop.step_value(x, current)
        y_next = params["Wout"] @ r_next + params["bout"]
        return x_next, (y_next, r_next)

    def rollout(self, params, input_sequence):
        x0 = jnp.zeros((self.config.n_hidden,), dtype=jnp.float32)

        def step(x, u_t):
            return self.one_step(params, x, u_t)

        _, (y, rates) = bst.transform.scan(step, x0, input_sequence)
        return y, rates

    def batch_rollout(self, params, input_sequences):
        ys, rates = jax.vmap(lambda seq: self.rollout(params, seq))(input_sequences)
        return ys, rates

    def summary(self, params):
        n_params = int(sum(np.prod(np.asarray(v).shape) for v in params.values()))
        print("ContextIntegrationRNN")
        print(f"  input dim  : {self.config.n_input}  [color, motion, color-context, motion-context]")
        print(f"  hidden dim : {self.config.n_hidden}")
        print(f"  output dim : {self.config.n_output}")
        print(f"  dt / tau   : {self.config.dt:.3f} / {self.config.tau:.3f} s")
        print(f"  parameters : {n_params:,}")
        for name, value in params.items():
            print(f"    {name:5s} {tuple(value.shape)}")

In [ ]:
def sample_context_integration_batch(key, batch_size=256, n_steps=75, noise_scale=0.20):
    """Generate synthetic trials following the context-dependent integration task."""
    k_ctx, k_sm, k_sc, k_mm, k_mc, k_nm, k_nc = jax.random.split(key, 7)

    ctx_motion = jax.random.bernoulli(k_ctx, 0.5, (batch_size,)).astype(jnp.float32)
    ctx_color = 1.0 - ctx_motion

    sign_m = 2.0 * jax.random.bernoulli(k_sm, 0.5, (batch_size,)).astype(jnp.float32) - 1.0
    sign_c = 2.0 * jax.random.bernoulli(k_sc, 0.5, (batch_size,)).astype(jnp.float32) - 1.0
    mag_levels = jnp.asarray([0.04, 0.08, 0.14, 0.20], dtype=jnp.float32)
    mag_m = mag_levels[jax.random.randint(k_mm, (batch_size,), 0, len(mag_levels))]
    mag_c = mag_levels[jax.random.randint(k_mc, (batch_size,), 0, len(mag_levels))]

    motion_offset = sign_m * mag_m
    color_offset = sign_c * mag_c
    motion = motion_offset[:, None] + noise_scale * jax.random.normal(k_nm, (batch_size, n_steps))
    color = color_offset[:, None] + noise_scale * jax.random.normal(k_nc, (batch_size, n_steps))

    inputs = jnp.stack([
        color,
        motion,
        jnp.repeat(ctx_color[:, None], n_steps, axis=1),
        jnp.repeat(ctx_motion[:, None], n_steps, axis=1),
    ], axis=-1)

    target_final = jnp.where(ctx_motion > 0.5, sign_m, sign_c)[:, None]
    target_initial = jnp.zeros_like(target_final)
    return inputs, target_initial, target_final


class TaskRNNTrainer:
    """BPTT trainer for the task RNN; loss is imposed at trial onset and offset."""

    def __init__(self, model, learning_rate=2e-3, l2=1e-5):
        self.model = model
        self.optimizer = optax.adam(learning_rate)
        self.l2 = l2

    def init_optimizer(self, params):
        return self.optimizer.init(params)

    def loss(self, params, inputs, target_initial, target_final):
        y, _ = self.model.batch_rollout(params, inputs)
        y0 = y[:, 0, :]
        yT = y[:, -1, :]
        mse = jnp.mean((y0 - target_initial) ** 2) + jnp.mean((yT - target_final) ** 2)
        reg = sum(jnp.mean(v ** 2) for v in params.values())
        return mse + self.l2 * reg, mse

    @brainx_jit(static_argnums=(0,))
    def train_step(self, params, opt_state, inputs, target_initial, target_final):
        (loss, mse), grads = jax.value_and_grad(self.loss, has_aux=True)(params, inputs, target_initial, target_final)
        updates, opt_state = self.optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss, mse

    def fit(self, params, key, n_steps=500, batch_size=256, n_time=75, log_every=100):
        opt_state = self.init_optimizer(params)
        history = []
        for step in range(1, n_steps + 1):
            key, subkey = jax.random.split(key)
            inputs, y0, yT = sample_context_integration_batch(subkey, batch_size=batch_size, n_steps=n_time)
            params, opt_state, loss, mse = self.train_step(params, opt_state, inputs, y0, yT)
            if step == 1 or step % log_every == 0:
                history.append((step, float(loss), float(mse)))
                print(f"step {step:04d} | task loss={float(loss):.4f} | task mse={float(mse):.4f}")
        return params, np.asarray(history)


task_config = TaskRateRNNConfig(n_hidden=100, dt=0.01, tau=0.05, gain=1.1)
task_model = ContextIntegrationRNN(task_config)
task_params = task_model.init_params(jax.random.PRNGKey(123))
task_model.summary(task_params)

task_trainer = TaskRNNTrainer(task_model, learning_rate=2e-3, l2=1e-5)
task_params, task_history = task_trainer.fit(
    task_params,
    key=jax.random.PRNGKey(321),
    n_steps=500,
    batch_size=256,
    n_time=75,
    log_every=100,
)

In [ ]:
def make_task_grid(n_steps=75, n_repeats=20, noise_scale=0.08):
    levels = np.array([-0.20, -0.14, -0.08, -0.04, 0.04, 0.08, 0.14, 0.20], dtype=np.float32)
    rows = []
    inputs = []
    rng = np.random.default_rng(0)
    for ctx_motion in [1.0, 0.0]:
        ctx_color = 1.0 - ctx_motion
        for m in levels:
            for c in levels:
                for _ in range(n_repeats):
                    motion = m + noise_scale * rng.standard_normal(n_steps)
                    color = c + noise_scale * rng.standard_normal(n_steps)
                    inp = np.stack([
                        color,
                        motion,
                        np.full(n_steps, ctx_color),
                        np.full(n_steps, ctx_motion),
                    ], axis=-1)
                    inputs.append(inp.astype(np.float32))
                    rows.append((ctx_motion, m, c))
    return jnp.asarray(np.stack(inputs)), np.asarray(rows), levels


grid_inputs, grid_meta, grid_levels = make_task_grid()
grid_y, grid_rates = task_model.batch_rollout(task_params, grid_inputs)
grid_choice = np.asarray(grid_y[:, -1, 0] > 0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
axes[0].plot(task_history[:, 0], task_history[:, 2], marker="o")
axes[0].set_xlabel("BPTT step")
axes[0].set_ylabel("task MSE")
axes[0].set_title("task RNN training")

for ctx_motion, ax, title in [(1.0, axes[1], "motion context"), (0.0, axes[2], "color context")]:
    for varying, label, color in [("motion", "motion evidence", "0.25"), ("color", "color evidence", "tab:blue")]:
        probs = []
        for lev in grid_levels:
            if varying == "motion":
                mask = (grid_meta[:, 0] == ctx_motion) & np.isclose(grid_meta[:, 1], lev)
            else:
                mask = (grid_meta[:, 0] == ctx_motion) & np.isclose(grid_meta[:, 2], lev)
            probs.append(np.mean(grid_choice[mask]))
        ax.plot(grid_levels, probs, marker="o", label=label, color=color)
    ax.axhline(0.5, color="0.6", lw=1, ls="--")
    ax.set_title(title)
    ax.set_xlabel("evidence")
    ax.set_ylabel("P(choice +1)")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(frameon=False, fontsize=8)
fig.tight_layout()

# Compare population trajectories from real PFC data and task RNN activity in PCA spaces.
# These PCA spaces are not the same coordinate system; the point is to show analogous population-dynamics views.
traj_specs = [
    {"ctx": 1, "m": -0.5, "c": 0.5, "label": "data motion ctx, m-", "color": "0.25"},
    {"ctx": 1, "m": 0.5, "c": -0.5, "label": "data motion ctx, m+", "color": "0.55"},
    {"ctx": -1, "m": -0.5, "c": 0.5, "label": "data color ctx, c+", "color": "tab:blue"},
    {"ctx": -1, "m": 0.5, "c": -0.5, "label": "data color ctx, c-", "color": "tab:cyan"},
]

def nearest_level(values, target):
    values = np.asarray(values)
    return values[np.argmin(np.abs(values - target))]

data_select = []
for spec in traj_specs:
    mval = nearest_level(cond_motion, spec["m"])
    cval = nearest_level(cond_color, spec["c"])
    idxs = np.where((cond_context == spec["ctx"]) & np.isclose(cond_motion, mval) & np.isclose(cond_color, cval))[0]
    if len(idxs) > 0:
        data_select.append(idxs[0])
data_select = np.asarray(data_select)

X_data_sel = np.transpose(X[:, :, data_select], (2, 1, 0))  # selected condition x time x unit
data_flat = X_data_sel.reshape(-1, X_data_sel.shape[-1])
data_pc = PCA(n_components=3).fit_transform(data_flat).reshape(X_data_sel.shape[0], X_data_sel.shape[1], 3)

rnn_select = []
rnn_specs = [(1.0, -0.20, 0.20), (1.0, 0.20, -0.20), (0.0, -0.20, 0.20), (0.0, 0.20, -0.20)]
for ctx_motion, m, c in rnn_specs:
    idxs = np.where((grid_meta[:, 0] == ctx_motion) & np.isclose(grid_meta[:, 1], m) & np.isclose(grid_meta[:, 2], c))[0]
    rnn_select.append(idxs[0])
rnn_select = np.asarray(rnn_select)
rates_sel = np.asarray(grid_rates[rnn_select])
rnn_flat = rates_sel.reshape(-1, rates_sel.shape[-1])
rnn_pc = PCA(n_components=3).fit_transform(rnn_flat).reshape(rates_sel.shape[0], rates_sel.shape[1], 3)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
for i, spec in enumerate(traj_specs[:len(data_select)]):
    axes[0].plot(data_pc[i, :, 0], data_pc[i, :, 1], "-o", ms=2.5, color=spec["color"], label=spec["label"])
axes[0].set_xlabel("PFC activity PC1")
axes[0].set_ylabel("PFC activity PC2")
axes[0].set_title("real PFC population trajectories")
axes[0].legend(fontsize=7, frameon=False)

rnn_labels = ["RNN motion ctx, m-", "RNN motion ctx, m+", "RNN color ctx, c+", "RNN color ctx, c-"]
rnn_colors = ["0.25", "0.55", "tab:blue", "tab:cyan"]
for i, label in enumerate(rnn_labels):
    axes[1].plot(rnn_pc[i, :, 0], rnn_pc[i, :, 1], "-o", ms=2.5, color=rnn_colors[i], label=label)
axes[1].set_xlabel("RNN activity PC1")
axes[1].set_ylabel("RNN activity PC2")
axes[1].set_title("task RNN internal trajectories")
axes[1].legend(fontsize=7, frameon=False)
fig.suptitle("Population-dynamics view: data PCA vs task-RNN PCA", y=1.02)
fig.tight_layout()


## 9. 用 BrainX/JAX 拟合低维 Linear Dynamical System

在 TDR 任务子空间中，可以先拟合一个简洁的 LDS baseline：

\[
z_{t+1}=A z_t + B u_c + b + \epsilon.
\]

这里的输入 \(u_c\) 是每个 condition 的 motion、color 和 context。我们用 ridge regression 拟合参数，再用 `brainstate.transform.jit` 编译 rollout 函数。

In [ ]:
def fit_lds_ridge(Z, conditions, dims=(0, 1, 2, 3), ridge=1e-3, test_stride=5):
    Y = np.transpose(Z[list(dims), :, :], (2, 1, 0))  # condition x time x dim
    Uc = np.column_stack([
        condition_array(conditions, "stim_dir"),
        condition_array(conditions, "stim_col2dir"),
        condition_array(conditions, "context"),
    ]).astype(float)
    Uc[:, 0] /= np.max(np.abs(Uc[:, 0]))
    Uc[:, 1] /= np.max(np.abs(Uc[:, 1]))

    n_cond, n_time, dim = Y.shape
    test_idx = np.arange(0, n_cond, test_stride)
    train_idx = np.setdiff1d(np.arange(n_cond), test_idx)

    rows = []
    targets = []
    for c in train_idx:
        for t in range(n_time - 1):
            rows.append(np.r_[Y[c, t], Uc[c], 1.0])
            targets.append(Y[c, t + 1])
    Phi = np.asarray(rows)
    Target = np.asarray(targets)

    reg = ridge * np.eye(Phi.shape[1])
    reg[-1, -1] = 0.0
    coef = np.linalg.solve(Phi.T @ Phi + reg, Phi.T @ Target)
    A = coef[:dim, :].T
    B = coef[dim:dim + Uc.shape[1], :].T
    b = coef[-1, :]
    return A, B, b, Uc, Y, train_idx, test_idx

A, B, b, Uc, Y_low, train_idx, test_idx = fit_lds_ridge(Z, conditions)
print("A shape:", A.shape, "B shape:", B.shape, "held-out conditions:", len(test_idx))


def _lds_rollout(A, B, b, z0, u_cond, n_steps):
    def step(z, _):
        z_next = A @ z + B @ u_cond + b
        return z_next, z_next
    _, zs = jax.lax.scan(step, z0, jnp.arange(n_steps - 1))
    return jnp.concatenate([z0[None, :], zs], axis=0)

lds_rollout = brainx_jit(_lds_rollout, static_argnames=("n_steps",))

A_j, B_j, b_j = map(jnp.asarray, (A, B, b))
preds = []
for c in test_idx:
    pred = lds_rollout(A_j, B_j, b_j, jnp.asarray(Y_low[c, 0]), jnp.asarray(Uc[c]), n_steps=Y_low.shape[1])
    preds.append(np.asarray(pred))
preds = np.stack(preds, axis=0)
truth = Y_low[test_idx]

r2 = 1.0 - np.sum((preds - truth) ** 2) / np.sum((truth - np.mean(truth, axis=(0, 1), keepdims=True)) ** 2)
print(f"held-out trajectory R^2: {r2:.3f}")
print("rollout compiled with:", "brainstate.transform.jit" if BRAINX_AVAILABLE else "jax.jit fallback")

In [ ]:
idx = 0
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].plot(truth[idx, :, 1], truth[idx, :, 0], "-o", label="data", ms=3)
axes[0].plot(preds[idx, :, 1], preds[idx, :, 0], "--o", label="LDS", ms=3)
axes[0].set_xlabel("motion axis")
axes[0].set_ylabel("choice axis")
axes[0].set_title("held-out trajectory")
axes[0].legend(frameon=False)

for d, name in enumerate(axis_names):
    axes[1].plot(tt, truth[idx, :, d], lw=2, label=f"data {name}")
    axes[1].plot(tt, preds[idx, :, d], "--", lw=1.5, label=f"pred {name}")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("TDR coordinate")
axes[1].set_title("LDS time courses")
axes[1].legend(fontsize=7, ncol=2, frameon=False)
fig.tight_layout()

## 10. 实战扩展：搭建 rate neuron / network，并用 BPTT 拟合 trace

这一节是对原文任务 RNN 思路的实战扩展：构建一个连续时间 firing-rate recurrent network，用于直接拟合真实神经群体活动的完整时间轨迹。

神经元动力学：

\[
\tau \dot{x} = -x + I(t), \qquad r=\phi(x)=\tanh(x)
\]

网络输入电流：

\[
I(t)=Jr(t)+W_{in}u_c+b
\]

线性读出：

\[
\hat{y}(t)=W_{out}r(t)+b_{out}
\]

- `RateNeuron`：定义连续时间 rate activation 和非线性 firing rate。
- `RateTraceNetwork`：定义 recurrent connectivity、condition input 和 neural activity readout。
- `RateBPTTTrainer`：对完整 trace 展开，通过 BPTT 优化网络参数。

训练目标是拟合整段 condition-averaged population trace，而不是只拟合最终选择或低维轨迹。


In [ ]:
from dataclasses import dataclass
import optax


@dataclass(frozen=True)
class RateNeuronConfig:
    n_hidden: int
    dt: float
    tau: float


class RateNeuron(bst.nn.Dynamics):
    """Continuous-time rate neuron population with r=tanh(x)."""

    def __init__(self, config: RateNeuronConfig):
        super().__init__(config.n_hidden)
        self.config = config

    def init_state(self, batch_size=None, **kwargs):
        shape = (self.config.n_hidden,) if batch_size is None else (batch_size, self.config.n_hidden)
        self.x = bst.HiddenState(jnp.zeros(shape, dtype=jnp.float32))

    def step_value(self, x, current):
        dx = (-x + current) / self.config.tau
        x_next = x + self.config.dt * dx
        r_next = jnp.tanh(x_next)
        return x_next, r_next

    def update(self, current):
        x_next, r_next = self.step_value(self.x.value, current)
        self.x.value = x_next
        return r_next


@dataclass(frozen=True)
class RateTraceNetworkConfig:
    n_input: int
    n_hidden: int
    n_output: int
    dt: float
    tau: float
    gain: float = 1.1


class RateTraceNetwork(bst.nn.Module):
    """Rate-based recurrent network for fitting neural population traces."""

    def __init__(self, config: RateTraceNetworkConfig):
        super().__init__()
        self.config = config
        self.pop = RateNeuron(RateNeuronConfig(config.n_hidden, config.dt, config.tau))

    def init_params(self, key):
        cfg = self.config
        k1, k2, k3 = jax.random.split(key, 3)
        return {
            "J": cfg.gain * jax.random.normal(k1, (cfg.n_hidden, cfg.n_hidden)) / jnp.sqrt(float(cfg.n_hidden)),
            "Win": 0.6 * jax.random.normal(k2, (cfg.n_hidden, cfg.n_input)) / jnp.sqrt(float(cfg.n_input)),
            "bias": jnp.zeros((cfg.n_hidden,), dtype=jnp.float32),
            "Wout": 0.05 * jax.random.normal(k3, (cfg.n_output, cfg.n_hidden)) / jnp.sqrt(float(cfg.n_hidden)),
            "bout": jnp.zeros((cfg.n_output,), dtype=jnp.float32),
        }

    def one_step(self, params, x, u_t):
        r = jnp.tanh(x)
        current = params["J"] @ r + params["Win"] @ u_t + params["bias"]
        x_next, r_next = self.pop.step_value(x, current)
        y_next = params["Wout"] @ r_next + params["bout"]
        return x_next, y_next

    def rollout(self, params, input_sequence):
        x0 = jnp.zeros((self.config.n_hidden,), dtype=jnp.float32)

        def step(x, u_t):
            return self.one_step(params, x, u_t)

        _, trace = bst.transform.scan(step, x0, input_sequence)
        return trace

    def batch_rollout(self, params, input_sequences):
        return jax.vmap(lambda seq: self.rollout(params, seq))(input_sequences)

    def n_parameters(self, params):
        return int(sum(np.prod(np.asarray(v).shape) for v in params.values()))

    def summary(self, params):
        print("RateTraceNetwork")
        print("  neuron     : RateNeuron(tau * dx/dt = -x + I, r=tanh(x))")
        print(f"  input dim  : {self.config.n_input}")
        print(f"  hidden dim : {self.config.n_hidden}")
        print(f"  output dim : {self.config.n_output}")
        print(f"  dt / tau   : {self.config.dt:.3f} / {self.config.tau:.3f} s")
        print(f"  parameters : {self.n_parameters(params):,}")
        for name, value in params.items():
            print(f"    {name:5s} {tuple(value.shape)}")


def build_condition_input_sequences(conditions, n_time):
    motion = condition_array(conditions, "stim_dir").astype(float)
    color = condition_array(conditions, "stim_col2dir").astype(float)
    context = condition_array(conditions, "context").astype(float)

    motion = motion / np.max(np.abs(motion))
    color = color / np.max(np.abs(color))
    ctx_color = (context == -1).astype(float)
    ctx_motion = (context == +1).astype(float)

    Uc4 = np.column_stack([color, motion, ctx_color, ctx_motion]).astype(np.float32)
    return np.repeat(Uc4[:, None, :], n_time, axis=1)


# Target trace: condition x time x unit.
Y_unit = np.transpose(X, (2, 1, 0)).astype(np.float32)
U_seq = build_condition_input_sequences(conditions, n_time=Y_unit.shape[1])

net_config = RateTraceNetworkConfig(
    n_input=U_seq.shape[-1],
    n_hidden=64,
    n_output=Y_unit.shape[-1],
    dt=float(np.median(np.diff(tt))),
    tau=0.20,
    gain=1.1,
)

rate_net = RateTraceNetwork(net_config)
params = rate_net.init_params(jax.random.PRNGKey(42))
rate_net.summary(params)

In [ ]:
class RateBPTTTrainer:
    """BPTT trainer for fitting complete neural population traces."""

    def __init__(self, model, learning_rate=3e-3, l2=1e-5):
        self.model = model
        self.optimizer = optax.adam(learning_rate)
        self.l2 = l2

    def init_optimizer(self, params):
        return self.optimizer.init(params)

    def loss(self, params, inputs, targets):
        pred = self.model.batch_rollout(params, inputs)
        mse = jnp.mean((pred - targets) ** 2)
        reg = sum(jnp.mean(v ** 2) for v in params.values())
        return mse + self.l2 * reg, mse

    @brainx_jit(static_argnums=(0,))
    def train_step(self, params, opt_state, inputs, targets):
        (loss, mse), grads = jax.value_and_grad(self.loss, has_aux=True)(params, inputs, targets)
        updates, opt_state = self.optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss, mse

    def fit(self, params, inputs, targets, n_steps=500, log_every=50):
        opt_state = self.init_optimizer(params)
        history = []
        for step in range(1, n_steps + 1):
            params, opt_state, loss, mse = self.train_step(params, opt_state, inputs, targets)
            if step == 1 or step % log_every == 0:
                history.append((step, float(loss), float(mse)))
                print(f"step {step:04d} | loss={float(loss):.4f} | mse={float(mse):.4f}")
        return params, np.asarray(history)


def r2_score_np(pred, target):
    return 1.0 - np.sum((pred - target) ** 2) / np.sum((target - np.mean(target, axis=(0, 1), keepdims=True)) ** 2)


# Reuse the LDS split so linear baseline and rate RNN are evaluated on the same held-out conditions.
train_idx_rnn = train_idx
test_idx_rnn = test_idx

x_train = jnp.asarray(U_seq[train_idx_rnn])
y_train = jnp.asarray(Y_unit[train_idx_rnn])
x_all = jnp.asarray(U_seq)

trainer = RateBPTTTrainer(rate_net, learning_rate=3e-3, l2=1e-5)
params, history = trainer.fit(params, x_train, y_train, n_steps=500, log_every=50)

pred_unit = np.asarray(rate_net.batch_rollout(params, x_all))
r2_train_unit = r2_score_np(pred_unit[train_idx_rnn], Y_unit[train_idx_rnn])
r2_test_unit = r2_score_np(pred_unit[test_idx_rnn], Y_unit[test_idx_rnn])

# Project model-predicted unit activity into the same TDR axes.
pred_tdr = np.einsum("ctu,ua->cta", pred_unit, axes_task)
true_tdr = np.transpose(Z, (2, 1, 0))
r2_test_tdr = r2_score_np(pred_tdr[test_idx_rnn], true_tdr[test_idx_rnn])

print(f"unit-space R^2 | train={r2_train_unit:.3f}, held-out={r2_test_unit:.3f}")
print(f"TDR-space held-out R^2={r2_test_tdr:.3f}")
print("BPTT train_step compiled with:", "bst.transform.jit" if BRAINX_AVAILABLE else "jax.jit fallback")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))

axes[0].plot(history[:, 0], history[:, 2], marker="o")
axes[0].set_xlabel("BPTT step")
axes[0].set_ylabel("trace MSE")
axes[0].set_title("rate network BPTT loss")

c = int(test_idx_rnn[0])
axes[1].plot(true_tdr[c, :, 1], true_tdr[c, :, 0], "-o", ms=3, label="data")
axes[1].plot(pred_tdr[c, :, 1], pred_tdr[c, :, 0], "--o", ms=3, label="rate network")
axes[1].set_xlabel("motion axis")
axes[1].set_ylabel("choice axis")
axes[1].set_title("held-out TDR trajectory")
axes[1].legend(frameon=False)

for d, name in enumerate(axis_names):
    axes[2].plot(tt, true_tdr[c, :, d], lw=2, label=f"data {name}")
    axes[2].plot(tt, pred_tdr[c, :, d], "--", lw=1.4, label=f"RNN {name}")
axes[2].set_xlabel("time (s)")
axes[2].set_ylabel("TDR coordinate")
axes[2].set_title("trace fitting in TDR coordinates")
axes[2].legend(fontsize=7, ncol=2, frameon=False)

fig.tight_layout()

### 10.1 真实神经活动 trace vs RNN 预测 trace

这里的拟合目标是神经活动本身：`Y_unit[condition, time, unit]`，即 condition-averaged、平滑、z-score 后的 PFC firing-rate trace。下面选一个 held-out condition，直接比较真实 unit trace 与 RNN 预测 trace。

In [ ]:
# Pick the held-out condition with the best unit-space prediction, so the example is readable.
unit_sse = np.sum((pred_unit[test_idx_rnn] - Y_unit[test_idx_rnn]) ** 2, axis=(1, 2))
unit_sst = np.sum((Y_unit[test_idx_rnn] - np.mean(Y_unit[train_idx_rnn], axis=(0, 1), keepdims=True)) ** 2, axis=(1, 2))
cond_r2 = 1.0 - unit_sse / unit_sst
best_local = int(np.nanargmax(cond_r2))
c = int(test_idx_rnn[best_local])

true_trace = Y_unit[c]       # time x unit
pred_trace = pred_unit[c]    # time x unit
unit_sse_each = np.sum((pred_trace - true_trace) ** 2, axis=0)
unit_sst_each = np.sum((true_trace - np.mean(true_trace, axis=0, keepdims=True)) ** 2, axis=0)
unit_r2 = 1.0 - unit_sse_each / np.maximum(unit_sst_each, 1e-8)
shown_units = np.argsort(unit_r2)[-6:][::-1]

cond_info = conditions[c]
print("held-out condition index:", c)
print("condition:", cond_info)
print(f"condition unit-space R^2: {cond_r2[best_local]:.3f}")
print("shown units:", shown_units.tolist())

fig, axes = plt.subplots(2, 3, figsize=(11, 5.8), sharex=True)
for ax, unit_id in zip(axes.ravel(), shown_units):
    ax.plot(tt, true_trace[:, unit_id], lw=2, label="data")
    ax.plot(tt, pred_trace[:, unit_id], "--", lw=2, label="RNN")
    ax.set_title(f"unit {unit_id} | R²={unit_r2[unit_id]:.2f}", fontsize=9)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("z-scored rate")
axes[0, 0].legend(frameon=False)
fig.suptitle("Held-out neural activity traces: data vs rate-network prediction", y=1.02)
fig.tight_layout()

fig, axes = plt.subplots(1, 3, figsize=(12.8, 4), sharey=True)
fig.subplots_adjust(left=0.07, right=0.88, bottom=0.16, top=0.82, wspace=0.18)
vmax = np.nanpercentile(np.abs(true_trace), 95)
for ax, mat, title in [
    (axes[0], true_trace.T, "data"),
    (axes[1], pred_trace.T, "rate RNN"),
    (axes[2], (pred_trace - true_trace).T, "prediction error"),
]:
    im = ax.imshow(mat, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax,
                   extent=[tt[0], tt[-1], true_trace.shape[1], 0])
    ax.set_title(title)
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("unit")
cbar_ax = fig.add_axes([0.91, 0.18, 0.018, 0.58])
fig.colorbar(im, cax=cbar_ax, label="z-scored rate")
fig.suptitle("Held-out condition: population trace heatmap", y=0.96)

## 11. 小结

本 notebook 展示了从真实神经记录到 rate-based population dynamics 建模的完整流程：

- 将单元记录整理为 condition-averaged population response，并进行平滑、标准化与可视化。
- 通过 PCA 与 TDR 提取 choice、motion、color 和 context 相关的低维任务轴。
- 构建任务型 firing-rate RNN，使网络学习 context-dependent integration，并可视化其行为曲线和内部轨迹。
- 在 TDR 空间中拟合 LDS baseline，用于刻画群体轨迹的低维线性动力学。
- 明确区分原文任务型 rate RNN 与本 notebook 的 trace-fitting 扩展；后者通过 BPTT 拟合完整 neural population trace。
- 对比真实 trace 与模型预测 trace，评估模型是否捕捉到任务相关群体活动结构。

该流程体现了 rate model 的核心价值：用相对简洁的动力学系统连接任务变量、神经群体活动和可检验的轨迹预测。
